# Getirme denemeleri — rerank, hibrit, ikisi birlikte

Düz RAG'in (%83,9) üzerine üç ağır yöntem deniyor: cross-encoder ile yeniden sıralama, BM25 + yoğun aramayı RRF ile birleştirme, ve ikisi birlikte.

**Sonuç:** tek başına rerank (p = 0,17) ve tek başına hibrit (p = 0,50) anlamlı değil; ikisi birlikte +5 puan (p = 0,007). Kazanan yapılandırma bu.

**Ön koşul:** 01 numaralı defterin hücreleri aynı oturumda çalışmış olmalı.

## 0 sft hala gerekli mi

`hucreler/02_retrieval_deneyleri/0_sft_hala_gerekli_mi.py`

In [ ]:
@torch.no_grad()
def uret_taban(prompts, max_new_tokens=64, bs=8, etiket=""):
    tok.padding_side = "left"
    outs = []
    with model.disable_adapter():
        for i in range(0, len(prompts), bs):
            b = prompts[i:i+bs]
            enc = tok(b, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
            g = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                               pad_token_id=tok.pad_token_id)
            outs += tok.batch_decode(g[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
            print(f"  {etiket}{min(i+bs,len(prompts))}/{len(prompts)}", end="\r")
    print()
    return outs

print("ham model + RAG (adapter kapali, ayni baglam)")
hamrag_outs = uret_taban(B_promptlar, 64, bs=8, etiket="[ham+RAG] ")
hamrag_acc, hamrag_pred = dogruluk(hamrag_outs, test_rows, "[ham+RAG] ")

print("ham model, RAG yok")
hamsade_outs = uret_taban([prompt(SYSTEM_SFT, r["question"]) for r in test_rows],
                          64, bs=16, etiket="[ham] ")
hamsade_acc, _ = dogruluk(hamsade_outs, test_rows, "[ham] ")

from math import comb
a = sum(x != r["letter"] and y == r["letter"] for x, y, r in zip(hamrag_pred, B_pred, test_rows))
b = sum(x == r["letter"] and y != r["letter"] for x, y, r in zip(hamrag_pred, B_pred, test_rows))
n = a + b
p = (sum(comb(n, k) for k in range(max(a, b), n + 1)) / 2**n * 2) if n else 1.0

print(f"""
{'='*58}
                     RAG YOK      RAG VAR     SFT KATKISI
ham model            {hamsade_acc:>6.1%}      {hamrag_acc:>6.1%}
SFT'li model         {sade_acc:>6.1%}      {B_acc:>6.1%}
                   {sade_acc-hamsade_acc:>+7.1%}    {B_acc-hamrag_acc:>+7.1%}
{'='*58}
RAG varken SFT: lehte {a} soru, aleyhte {b}, p = {min(p,1.0):.3f}
{'  -> SFT hala katki sagliyor' if p < 0.05 and B_acc > hamrag_acc else '  -> SFT katkisi RAG tarafindan ikame edilmis'}""")

## 1 reranker

`hucreler/02_retrieval_deneyleri/1_reranker.py`

In [ ]:
!pip install -q FlagEmbedding 2>/dev/null || None

import torch, gc
from FlagEmbedding import FlagReranker

RERANK_MODEL = "BAAI/bge-reranker-v2-m3"
GETIR_K, TUT_N, LIMIT = 30, 6, 6000

print("reranker yukleniyor")
reranker = FlagReranker(RERANK_MODEL, use_fp16=True, devices=["cuda:0"])

print(f"\nk={GETIR_K} ile genis getirme ({len(test_rows)} soru)")
eski_k = retriever.child_retriever.search_kwargs.get("k", settings.retrieval_k)
retriever.child_retriever.search_kwargs["k"] = GETIR_K
genis_docs = []
for i, r in enumerate(test_rows, 1):
    genis_docs.append(retriever.invoke(r["question"]))
    print(f"  {i}/{len(test_rows)}", end="\r")
retriever.child_retriever.search_kwargs["k"] = eski_k
print(f"\nortalama aday belge: {sum(len(d) for d in genis_docs)/len(genis_docs):.1f}")

print("\nyeniden siralama")
R_docs = []
for i, (r, docs) in enumerate(zip(test_rows, genis_docs), 1):
    if len(docs) <= TUT_N:
        R_docs.append(docs)
    else:
        ciftler = [[r["question"], d.page_content[:2000]] for d in docs]
        skor = reranker.compute_score(ciftler, normalize=True)
        sirali = sorted(zip(skor, range(len(docs))), reverse=True)
        R_docs.append([docs[j] for _, j in sirali[:TUT_N]])
    print(f"  {i}/{len(test_rows)}", end="\r")
print()

R_pr = [prompt(SYSTEM_RAG, f"KAYNAK METİN:\n{baglam_kur(d, LIMIT)}\n\n---\n\nSORU:\n{r['question']}")
        for r, d in zip(test_rows, R_docs)]
R_outs = uret(R_pr, 64, bs=4, etiket="[rerank] ")
R_acc, R_pred = dogruluk(R_outs, test_rows, "[rerank] ")
print(f"\nRAG duz {B_acc:.1%}  ->  rerank {R_acc:.1%}  ({R_acc-B_acc:+.1%})")

del reranker; gc.collect(); torch.cuda.empty_cache()

## 2 hibrit bm25

`hucreler/02_retrieval_deneyleri/2_hibrit_bm25.py`

In [ ]:
!pip install -q rank_bm25 2>/dev/null || None

import re, unicodedata
from rank_bm25 import BM25Okapi
from scripts.indexing.loader import load_markdown_documents
from scripts.indexing.chunker import ChunkingConfig, chunk_parent_child_documents

LIMIT, TUT_N = 6000, 6

print("parent chunk'lar yeniden uretiliyor (Qdrant'a dokunulmuyor)")
belgeler = load_markdown_documents(DATA)
parents, _ = chunk_parent_child_documents(
    belgeler,
    parent_config=ChunkingConfig(max_tokens=settings.parent_chunk_size_tokens, overlap_tokens=0),
    child_config=ChunkingConfig(max_tokens=settings.chunk_size_tokens,
                                overlap_tokens=settings.chunk_overlap_tokens),
)
print(f"parent: {len(parents)}")

def jeton(s):
    s = unicodedata.normalize("NFD", s.lower())
    s = "".join(c for c in s if not unicodedata.combining(c))
    return [w for w in re.findall(r"[a-z0-9]+", s) if len(w) > 2]

print("BM25 kuruluyor")
korpus = [jeton(p.page_content) for p in parents]
bm25 = BM25Okapi(korpus)

def rrf(siralar, k=60):
    """Reciprocal Rank Fusion — birden cok siralamayi birlestirir."""
    puan = {}
    for sira in siralar:
        for yer, kimlik in enumerate(sira):
            puan[kimlik] = puan.get(kimlik, 0.0) + 1.0 / (k + yer + 1)
    return [kimlik for kimlik, _ in sorted(puan.items(), key=lambda x: -x[1])]

icerik_idx = {p.page_content: i for i, p in enumerate(parents)}

print(f"\nhibrit getirme ({len(test_rows)} soru)")
eski_k = retriever.child_retriever.search_kwargs.get("k", settings.retrieval_k)
retriever.child_retriever.search_kwargs["k"] = 20
H_docs = []
for i, r in enumerate(test_rows, 1):
    yogun = retriever.invoke(r["question"])
    yogun_idx = [icerik_idx.get(d.page_content) for d in yogun]
    yogun_idx = [x for x in yogun_idx if x is not None]
    seyrek_idx = list(bm25.get_top_n(jeton(r["question"]), range(len(parents)), n=20))
    birlesik = rrf([yogun_idx, seyrek_idx])[:TUT_N]
    H_docs.append([parents[j] for j in birlesik])
    print(f"  {i}/{len(test_rows)}", end="\r")
retriever.child_retriever.search_kwargs["k"] = eski_k
print()

ortak = sum(len(set(icerik_idx.get(d.page_content) for d in H_docs[i])) for i in range(len(H_docs)))
print(f"ortalama belge: {ortak/len(H_docs):.1f}")

H_pr = [prompt(SYSTEM_RAG, f"KAYNAK METİN:\n{baglam_kur(d, LIMIT)}\n\n---\n\nSORU:\n{r['question']}")
        for r, d in zip(test_rows, H_docs)]
H_outs = uret(H_pr, 64, bs=4, etiket="[hibrit] ")
H_acc, H_pred = dogruluk(H_outs, test_rows, "[hibrit] ")
print(f"\nRAG duz {B_acc:.1%}  ->  hibrit {H_acc:.1%}  ({H_acc-B_acc:+.1%})")

## 3 hibrit plus rerank

`hucreler/02_retrieval_deneyleri/3_hibrit_plus_rerank.py`

In [ ]:
import torch, gc
from FlagEmbedding import FlagReranker

LIMIT, ADAY_N, TUT_N = 6000, 30, 6
reranker = FlagReranker("BAAI/bge-reranker-v2-m3", use_fp16=True, devices=["cuda:0"])

print(f"hibrit aday havuzu ({ADAY_N}) + yeniden siralama")
eski_k = retriever.child_retriever.search_kwargs.get("k", settings.retrieval_k)
retriever.child_retriever.search_kwargs["k"] = ADAY_N
HR_docs = []
for i, r in enumerate(test_rows, 1):
    yogun = retriever.invoke(r["question"])
    yogun_idx = [icerik_idx.get(d.page_content) for d in yogun]
    yogun_idx = [x for x in yogun_idx if x is not None]
    seyrek_idx = bm25.get_top_n(jeton(r["question"]), range(len(parents)), n=ADAY_N)
    aday = [parents[j] for j in rrf([yogun_idx, list(seyrek_idx)])[:ADAY_N]]
    if len(aday) <= TUT_N:
        HR_docs.append(aday)
    else:
        skor = reranker.compute_score([[r["question"], d.page_content[:2000]] for d in aday],
                                      normalize=True)
        sirali = sorted(zip(skor, range(len(aday))), reverse=True)
        HR_docs.append([aday[j] for _, j in sirali[:TUT_N]])
    print(f"  {i}/{len(test_rows)}", end="\r")
retriever.child_retriever.search_kwargs["k"] = eski_k
print()

HR_pr = [prompt(SYSTEM_RAG, f"KAYNAK METİN:\n{baglam_kur(d, LIMIT)}\n\n---\n\nSORU:\n{r['question']}")
         for r, d in zip(test_rows, HR_docs)]
HR_outs = uret(HR_pr, 64, bs=4, etiket="[hibrit+rerank] ")
HR_acc, HR_pred = dogruluk(HR_outs, test_rows, "[hibrit+rerank] ")
print(f"\nRAG duz {B_acc:.1%}  ->  hibrit+rerank {HR_acc:.1%}  ({HR_acc-B_acc:+.1%})")

del reranker; gc.collect(); torch.cuda.empty_cache()

## 4 ozet

`hucreler/02_retrieval_deneyleri/4_ozet.py`

In [ ]:
import json
from math import comb

def mcnemar(p1, p2):
    a = sum(x != r["letter"] and y == r["letter"] for x, y, r in zip(p1, p2, test_rows))
    b = sum(x == r["letter"] and y != r["letter"] for x, y, r in zip(p1, p2, test_rows))
    n = a + b
    p = (sum(comb(n, k) for k in range(max(a, b), n + 1)) / 2**n * 2) if n else 1.0
    return a, b, min(p, 1.0)

adaylar = [("RAG yok", sade_acc, None), ("RAG duz (k=6, 6000)", B_acc, B_pred)]
for ad, deg in [("rerank", "R"), ("hibrit", "H"), ("hibrit + rerank", "HR")]:
    if f"{deg}_acc" in globals():
        adaylar.append((ad, globals()[f"{deg}_acc"], globals()[f"{deg}_pred"]))

print("="*64)
for ad, acc, _ in adaylar:
    print(f"{ad:<26}{acc:>7.1%}")
print("="*64)
print("\nRAG duz'e gore (McNemar):")
for ad, acc, pred in adaylar[2:]:
    a, b, p = mcnemar(B_pred, pred)
    print(f"  {ad:<22} {acc-B_acc:+.1%}  |  lehte {a} aleyhte {b}  |  p = {p:.3f}"
          + ("  ANLAMLI" if p < 0.05 else ""))

OUT = "/content/drive/MyDrive/tus_sft/rag_sonuc"
os.makedirs(OUT, exist_ok=True)
with open(f"{OUT}/agir_deneyler.json", "w", encoding="utf-8") as f:
    json.dump({ad: acc for ad, acc, _ in adaylar}, f, ensure_ascii=False, indent=2)
print("\nkaydedildi:", OUT)